# 02 — Cross-Encoder Reranker Fine-Tuning (BGE-reranker-v2-m3 + LoRA)

This notebook fine-tunes `BAAI/bge-reranker-v2-m3` — a state-of-the-art multilingual cross-encoder — on Turkish legal `(query, passage)` pairs.

**Approach.** Rather than training a reranker from scratch, we domain-adapt a strong pretrained cross-encoder with LoRA (`r=16`) targeting attention `query / key / value` projections. The classifier head is kept fully trainable (`modules_to_save=['classifier']`). This keeps trainable parameters to ~10M out of 568M, fits a T4 comfortably, and reduces overfitting risk on a small in-domain dataset.

**Training data.** `embed_triplets_v2.jsonl` — 3,026 anchor / positive / hard-negatives tuples, where negatives are mined from the union of BM25 and dense (e5) top-K plus a same-instrument boost. Each tuple expands into 1 positive + 7 negative pairs, yielding ~24K binary classification pairs.

**Validation.** Difficulty-stratified 10% holdout (deterministic seed). Two metrics tracked per epoch:
- `val_AP` — average precision on the binary task (primary, used for best-checkpoint selection).
- `Mini gate R@10 / R@5 / MRR` — synthetic retrieval gate: 50 random val queries scored against a pool of (positive + hard negatives + 42 random distractors).

**Inputs (Kaggle).**
- `embed_triplets_v2.jsonl`

**Output.** `bge-reranker-tr-legal-v2/` — LoRA adapter (~30 MB) + tokenizer + `best_meta.json` + `training_history.json`.

**Wall time.** ~2–2.5 hours on T4×2.

---

**v3 (2026-05-22):** retrained on cleaned corpus + listwise softmax CE.

- Triplets: `embed_triplets_v3.jsonl` (2,861 anchors x 10 hard negatives, isolation-clean)
- Loss: pointwise BCE -> **listwise softmax cross-entropy** over (positive, K hard-negatives) - FlagEmbedding recipe
- Best-epoch criterion: val_AP -> **gate MRR**
- LR: 1e-5 -> **5e-6**; epochs 3 -> **2**
- Output: `bge-reranker-tr-legal-v3`

In [ ]:
!pip install -q sentence-transformers==3.3.1 transformers==4.45.2 accelerate peft==0.13.2

In [ ]:
import json, random, math, os, torch, numpy as np
import torch.nn.functional as F
from pathlib import Path
from collections import defaultdict
from sentence_transformers import CrossEncoder
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from torch.utils.data import Dataset, DataLoader
from transformers import get_cosine_schedule_with_warmup

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Auto-detect dataset mount (works on any Kaggle account)
_candidates = list(Path('/kaggle/input').rglob('embed_triplets_v3.jsonl'))
assert _candidates, 'embed_triplets_v3.jsonl not found under /kaggle/input — attach the dataset!'
TRIPLETS = _candidates[0]
INPUT_DIR = TRIPLETS.parent
print('Found:', TRIPLETS)
OUTPUT_DIR = Path('/kaggle/working/bge-reranker-tr-legal-v3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = 'BAAI/bge-reranker-v2-m3'
QUERY_BATCH = 4          # T4 OOM at 8 — reduced to 4
GRAD_ACCUM = 2          # effective batch = 8 (same as v2 design)
EPOCHS = 2
LR = 5e-6
MAX_LEN = 512
WARMUP_RATIO = 0.06

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ['query', 'key', 'value']

GATE_QUERIES = 50
GATE_DISTRACTORS = 42

print('CUDA:', torch.cuda.is_available(), torch.cuda.device_count(), 'GPUs')


## 1. Data: triplets as listwise candidate lists

Each row is `(query, positive, neg1..neg10)`. Validation split: 10% per difficulty bucket (stratified, seed=42). No pair-expansion — listwise loss consumes the whole candidate list per query.


In [ ]:
by_diff = defaultdict(list)
all_rows = []
with TRIPLETS.open(encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        all_rows.append(r)
        by_diff[r.get('difficulty', 'unknown')].append(r)

print(f'Toplam triplet: {len(all_rows)}')
for d, rows in by_diff.items():
    print(f'  {d}: {len(rows)}')

val_rows, train_rows = [], []
rng = random.Random(SEED)
for d, rows in by_diff.items():
    shuf = rows.copy(); rng.shuffle(shuf)
    n_val = max(1, len(shuf) // 10)
    val_rows.extend(shuf[:n_val])
    train_rows.extend(shuf[n_val:])
rng.shuffle(train_rows)
rng.shuffle(val_rows)
print(f'\nTrain triplets: {len(train_rows)} | Val triplets: {len(val_rows)}')

sizes = [1 + len(r['negatives']) for r in train_rows]
assert len(set(sizes)) == 1, f'Non-uniform candidate counts: {set(sizes)}'
K_PLUS_1 = sizes[0]
print(f'Candidates per query (1 pos + N negs): {K_PLUS_1}')

with open(OUTPUT_DIR / 'val_triplets.jsonl', 'w', encoding='utf-8') as f:
    for r in val_rows:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')


## 2. Model construction with LoRA

`BGE-reranker-v2-m3` is an `XLMRobertaXLForSequenceClassification`. We load it through `CrossEncoder`, then wrap the underlying HuggingFace model with PEFT. Only the LoRA adapter and the classifier head receive gradients.

In [ ]:
model = CrossEncoder(BASE_MODEL, num_labels=1, max_length=MAX_LEN, device='cuda')

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias='none',
    modules_to_save=['classifier'],  # classifier head fully trainable
)
model.model = get_peft_model(model.model, lora_cfg)
model.model.print_trainable_parameters()
model.model.to('cuda')
# Enable gradient checkpointing to fit T4 VRAM (trades compute for memory)
model.model.gradient_checkpointing_enable()
model.model.enable_input_require_grads()  # required for grad ckpt with PEFT

## 3. Synthetic retrieval gate

End-of-epoch gate that mimics the downstream usage. For each of 50 sampled validation queries we build a candidate pool of `1 positive + all hard negatives + 42 random distractors`, score it with the current reranker, and record the rank of the positive. Reports R@1 / R@5 / R@10 / MRR / mean-rank.

In [ ]:
GATE_QUERIES = min(50, len(val_rows))
GATE_DISTRACTORS = 42  # so pool ≈ 50

gate_subset = val_rows[:GATE_QUERIES]
# Build a global distractor pool from all val passages (excludes queries' positives)
all_val_passages = []
for r in val_rows:
    all_val_passages.append((r['positive_doc_id'], r['positive']))
    for did, t in zip(r['negative_doc_ids'], r['negatives']):
        all_val_passages.append((did, t))

def run_mini_gate(ce_model):
    rng2 = random.Random(SEED)
    ranks = []
    for q in gate_subset:
        pos_id = q['positive_doc_id']
        # pool: positive + its hard negs + random distractors
        pool = [(pos_id, q['positive'])]
        for did, t in zip(q['negative_doc_ids'], q['negatives']):
            pool.append((did, t))
        # distractors
        distractors = rng2.sample(
            [p for p in all_val_passages if p[0] != pos_id],
            min(GATE_DISTRACTORS, len(all_val_passages) - 1),
        )
        pool.extend(distractors)
        # dedup by doc_id
        seen = set(); deduped = []
        for did, t in pool:
            if did in seen: continue
            seen.add(did); deduped.append((did, t))
        pool = deduped
        scores = ce_model.predict([[q['anchor'], t] for _, t in pool], show_progress_bar=False, batch_size=32)
        order = np.argsort(-np.asarray(scores))
        rank = next((i + 1 for i, j in enumerate(order) if pool[j][0] == pos_id), len(pool) + 1)
        ranks.append(rank)
    ranks = np.array(ranks)
    return {
        'R@1': float((ranks <= 1).mean()),
        'R@5': float((ranks <= 5).mean()),
        'R@10': float((ranks <= 10).mean()),
        'MRR': float((1.0 / ranks).mean()),
        'mean_rank': float(ranks.mean()),
    }

## 4. Listwise training loop (softmax CE) with gate-MRR best-checkpoint selection

For each batch of `B` queries we forward `B x (K+1)` `(query, candidate)` pairs through the cross-encoder, reshape logits to `(B, K+1)`, apply softmax, and minimize cross-entropy with target index 0 (positive always first). This is the FlagEmbedding listwise recipe — it teaches "positive ranks above all other candidates" rather than the pointwise "is this candidate relevant 0/1" signal, which is mismatched to ranking and produced the v2 regression (A4 < A3 in the first ablation round).

Best-epoch criterion is **gate MRR** (mini retrieval gate, mimics downstream usage), replacing `val_AP` which correlated poorly with downstream ranking quality.


In [ ]:
class ListwiseDataset(Dataset):
    def __init__(self, triplets):
        self.triplets = triplets
    def __len__(self):
        return len(self.triplets)
    def __getitem__(self, i):
        r = self.triplets[i]
        return {
            'query': r['anchor'],
            'candidates': [r['positive']] + r['negatives'],
        }

def listwise_collate(batch, tokenizer, max_len, k_plus_1):
    queries, cands = [], []
    for item in batch:
        assert len(item['candidates']) == k_plus_1, 'candidate count drift'
        for c in item['candidates']:
            queries.append(item['query'])
            cands.append(c)
    enc = tokenizer(queries, cands, padding=True, truncation=True,
                    max_length=max_len, return_tensors='pt')
    return enc, len(batch)

tokenizer = model.tokenizer

train_loader = DataLoader(
    ListwiseDataset(train_rows),
    batch_size=QUERY_BATCH, shuffle=True, num_workers=2, drop_last=True,
    collate_fn=lambda b: listwise_collate(b, tokenizer, MAX_LEN, K_PLUS_1),
)

device = 'cuda'
model.model.to(device)

trainable = [p for p in model.model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.01)

num_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
num_warmup = int(WARMUP_RATIO * num_steps)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup, num_training_steps=num_steps,
)
print(f'Train steps: {num_steps}, warmup: {num_warmup}')

scaler = torch.cuda.amp.GradScaler()
best_mrr = -1.0
best_epoch = -1
history = []
global_step = 0

for epoch in range(EPOCHS):
    print(f'\n{"="*60}\nEPOCH {epoch + 1}/{EPOCHS}\n{"="*60}')
    model.model.train()
    running_loss = 0.0
    n_batches = 0
    accum_count = 0
    for enc, B in train_loader:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.cuda.amp.autocast():
            out = model.model(**enc)
            logits = out.logits.squeeze(-1).view(B, K_PLUS_1)
            target = torch.zeros(B, dtype=torch.long, device=device)
            loss = F.cross_entropy(logits, target) / GRAD_ACCUM
        scaler.scale(loss).backward()
        accum_count += 1
        if accum_count % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            if global_step % 50 == 0:
                print(f'  step {global_step:5d} | loss {running_loss/max(1,n_batches):.4f} | lr {scheduler.get_last_lr()[0]:.2e}')
        running_loss += loss.item() * GRAD_ACCUM  # undo the division for logging
        n_batches += 1
    avg_loss = running_loss / max(1, n_batches)

    model.model.eval()
    with torch.no_grad():
        gate = run_mini_gate(model)
    history.append({'epoch': epoch + 1, 'train_loss': avg_loss, **gate})
    print(f'EPOCH {epoch + 1} | train_loss={avg_loss:.4f} | gate R@10={gate["R@10"]:.3f} R@5={gate["R@5"]:.3f} MRR={gate["MRR"]:.4f}')

    if gate['MRR'] > best_mrr:
        best_mrr = gate['MRR']
        best_epoch = epoch + 1
        model.model.save_pretrained(str(OUTPUT_DIR / 'lora_adapter'))
        model.tokenizer.save_pretrained(str(OUTPUT_DIR))
        with open(OUTPUT_DIR / 'best_meta.json', 'w', encoding='utf-8') as f:
            json.dump({'best_epoch': best_epoch, 'gate_mrr': best_mrr, 'base_model': BASE_MODEL,
                       'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA, 'max_len': MAX_LEN,
                       'lr': LR, 'epochs': EPOCHS, 'query_batch': QUERY_BATCH,
                       'k_plus_1': K_PLUS_1, 'loss': 'listwise_softmax_ce',
                       'gate': gate, 'final_train_loss': avg_loss}, f, ensure_ascii=False, indent=2)
        print(f'  -> NEW BEST (MRR {best_mrr:.4f}), saved to {OUTPUT_DIR}')

with open(OUTPUT_DIR / 'training_history.json', 'w', encoding='utf-8') as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print(f'\n{"="*60}\nBest epoch: {best_epoch} | gate MRR={best_mrr:.4f}\n{"="*60}')
for h in history:
    print(h)


## 5. Smoke test — load the LoRA adapter and score a sample query

In [ ]:
from peft import PeftModel

test = CrossEncoder(BASE_MODEL, num_labels=1, max_length=MAX_LEN, device='cuda')
test.model = PeftModel.from_pretrained(test.model, str(OUTPUT_DIR / 'lora_adapter'))
test.model.eval()

query = 'Cumhurbaşkanına hakaret cezası nedir?'
candidates = [
    'Cumhurbaşkanına hakaret eden kişi bir yıldan dört yıla kadar hapis cezası alır.',
    'Tacir her türlü borcu için iflasa tabidir.',
    'Bir insanı kasten öldüren kişi müebbet hapis cezası alır.',
    'Hakaret suçunun cezası üç aydan iki yıla kadar hapis veya adli para cezasıdır.',
]
scores = test.predict([[query, c] for c in candidates])
for c, s in sorted(zip(candidates, scores), key=lambda x: -x[1]):
    print(f'{s:+.3f}  {c[:80]}')

## 6. Export

Download `bge-reranker-tr-legal-v3/` from the Kaggle Output panel. The adapter is loaded at inference time on top of the same `BAAI/bge-reranker-v2-m3` base model.

After download, place under `data/models/bge-reranker-tr-legal-v3/`.
